# RAG-NetTwin: GPU-Accelerated Vector DB Re-index

Re-builds the ChromaDB vector store from `all_chunks.json` using GPU for massive speedup.

**On a T4 GPU**: ~2-3 minutes for 10K chunks (vs ~2 hours on CPU).

**Before running**: Upload your `all_chunks.json` and `graphify-docs/` folder to Google Drive, then mount it below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# --- CONFIG ---
# Path to your data directory in Google Drive
# Place all_chunks.json and graphify-docs/ under this path
DRIVE_DATA_DIR = "/content/drive/MyDrive/RAG-NetTwin/data"

# Where to save the built ChromaDB (persists across sessions)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/RAG-NetTwin/chroma_db"

In [ ]:
!pip install chromadb sentence-transformers==3.3.1 rich tqdm --quiet
!pip install "networkx>=3.0" --quiet

In [ ]:
import json
import os
import shutil
import hashlib
import time
from pathlib import Path
from typing import List, Dict, Any, Optional, cast

import torch
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
console.print(Panel(f"[bold green]Device: {DEVICE}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}[/bold green]"))

In [ ]:
# Copy data from Drive to local runtime (faster I/O)
DATA_DIR = Path("/content/data")
CHROMA_DIR = DATA_DIR / "chroma_db"
CHUNKS_FILE = DATA_DIR / "processed_chunks" / "all_chunks.json"
GRAPHIFY_DIR = DATA_DIR / "graphify-docs"

if CHUNKS_FILE.exists():
    console.print(f"[green]Data already at {CHUNKS_FILE}[/green]")
else:
    DRIVE_CHUNKS = Path(DRIVE_DATA_DIR) / "processed_chunks" / "all_chunks.json"
    DRIVE_GRAPHIFY = Path(DRIVE_DATA_DIR) / ".." / "graphify-docs"

    if not DRIVE_CHUNKS.exists():
        console.print(f"[bold red]File not found: {DRIVE_CHUNKS}[/bold red]")
        console.print("Please upload all_chunks.json to your Drive and update DRIVE_DATA_DIR.")
        raise FileNotFoundError(f"{DRIVE_CHUNKS} not found")

    CHUNKS_FILE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_CHUNKS, CHUNKS_FILE)
    console.print(f"[green]Copied {DRIVE_CHUNKS} -> {CHUNKS_FILE}[/green]")

    if DRIVE_GRAPHIFY.exists():
        shutil.copytree(DRIVE_GRAPHIFY, GRAPHIFY_DIR, dirs_exist_ok=True)
        console.print(f"[green]Copied graphify-docs -> {GRAPHIFY_DIR}[/green]")

if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
    console.print("[dim]Cleared existing chroma_db[/dim]")
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
MODEL_NAME = "ibm-granite/granite-embedding-311m-multilingual-r2"
RERANKER_NAME = "Alibaba-NLP/gte-reranker-modernbert-base"

console.print(f"[bold blue]Loading embedding model: {MODEL_NAME}...[/bold blue]")
t0 = time.time()
model = SentenceTransformer(MODEL_NAME, device=DEVICE, trust_remote_code=True)
if "jina" in MODEL_NAME.lower():
    model.default_task = "retrieval"
console.print(f"[green]Model loaded in {time.time()-t0:.1f}s[/green]")

console.print(f"[bold blue]Loading reranker: {RERANKER_NAME}...[/bold blue]")
from sentence_transformers import CrossEncoder
t0 = time.time()
reranker = CrossEncoder(RERANKER_NAME, device=DEVICE, trust_remote_code=True)
console.print(f"[green]Reranker loaded in {time.time()-t0:.1f}s[/green]")

In [ ]:
with open(CHUNKS_FILE, "r") as f:
    chunks = json.load(f)
console.print(f"[bold]Loaded {len(chunks)} total chunks[/bold]")

# Quality Gate: filter noise and boilerplate
boilerplate_keywords = [
    "copyright", "all rights reserved", "table of contents",
    "preface", "intended audience", "document history",
    "legal notice", "trademark", "proprietary information"
]

filtered = []
for c in chunks:
    text = c.get("text", "").lower()
    tier = c.get("tier", 1)
    if tier == 1:
        if len(text) > 40:
            filtered.append(c)
        continue
    if len(text) < 150:
        continue
    if any(kw in text for kw in boilerplate_keywords):
        continue
    filtered.append(c)

removed = len(chunks) - len(filtered)
console.print(f"[green]Quality Gate: Kept {len(filtered)} chunks, removed {removed}[/green]")
chunks = filtered
del filtered

In [ ]:
client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False)
)

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=MODEL_NAME, device=DEVICE
)

collection = client.create_collection(
    name="network_docs",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_fn,
)

# Also create secondary collections
client.create_collection(name="intent_few_shot", metadata={"hnsw:space": "cosine"})
client.create_collection(name="intent_cache", metadata={"hnsw:space": "cosine"})
client.create_collection(name="gold_configs", metadata={"hnsw:space": "cosine"})
client.create_collection(name="intent_parsing_cache", metadata={"hnsw:space": "cosine"})

console.print(f"[green]ChromaDB initialized at {CHROMA_DIR}[/green]")

In [ ]:
BATCH_SIZE = 500  # outer chunks per batch
ENCODE_BATCH_SIZE = 32  # inner encode batch (avoids OOM with long texts)

# CUDA warmup: encode a single text to trigger compilation before the main loop
console.print("[dim]Warming up CUDA...[/dim]")
_ = model.encode([doc_prefix + chunks[0]["text"]], convert_to_numpy=True, normalize_embeddings=True)
console.print("[green]CUDA ready. Starting bulk index...[/green]")

# Sort chunks by length descending to keep per-encode-batch padding minimal
chunks.sort(key=lambda c: -len(c.get("text", "")))

is_granite = "granite" in MODEL_NAME.lower()
doc_prefix = (
    "Instruction: Given a network documentation chunk, extract and embed it for retrieval.\nChunk: "
    if is_granite else ""
)

total_added = 0
t_start = time.time()

for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Bulk Indexing", unit="batch"):
    batch = chunks[i : i + BATCH_SIZE]

    batch_texts = []
    batch_metas = []
    batch_ids = []

    for chunk in batch:
        text = chunk.get("text", "")
        metas = {
            "tier": chunk.get("tier", 1),
            "vendor": chunk.get("vendor", "unknown"),
            "os_version": chunk.get("os_version", "unknown"),
            "command_hierarchy": json.dumps(chunk.get("command_hierarchy", [])),
            "context_tags": json.dumps(chunk.get("context_tags", [])),
            "source_file": chunk.get("source_file", "unknown"),
            "page_range": json.dumps(chunk.get("page_range", [0, 0])),
        }
        cid = chunk.get("id", hashlib.md5(text.encode()).hexdigest())

        batch_texts.append(text)
        batch_metas.append(metas)
        batch_ids.append(cid)

    processed_texts = [doc_prefix + t for t in batch_texts]

    embeddings = model.encode(
        processed_texts,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).tolist()

    collection.add(
        documents=batch_texts,
        embeddings=embeddings,
        metadatas=cast(Any, batch_metas),
        ids=batch_ids,
    )

    total_added += len(batch_texts)
    elapsed = time.time() - t_start
    rate = total_added / elapsed
    tqdm.write(f"  Added {total_added}/{len(chunks)} chunks | {rate:.0f} chunks/s | {elapsed:.0f}s elapsed")

total_time = time.time() - t_start
console.print(Panel(
    f"[bold green]✓ Indexed {total_added} chunks in {total_time:.0f}s ({total_added/total_time:.0f} chunks/s)[/bold green]",
    border_style="green"
))

In [ ]:
count = collection.count()
console.print(f"ChromaDB collection 'network_docs' has [bold]{count}[/bold] chunks")

# Quick sanity query
test_query = "configure OSPF area 0"
query_prefix = "Instruction: Given a network engineering query, retrieve the most relevant command documentation.\nQuery: " if is_granite else ""
q_emb = model.encode([query_prefix + test_query], convert_to_numpy=True, normalize_embeddings=True).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

table = Table(title="Sample Results: 'configure OSPF area 0'")
table.add_column("ID", style="cyan")
table.add_column("Distance", justify="right", style="green")
table.add_column("Preview", style="white")
for i in range(len(results["ids"][0])):
    rid = results["ids"][0][i]
    dist = results["distances"][0][i]
    preview = (results["documents"][0][i] or "")[:80] + "..."
    table.add_row(rid, f"{dist:.4f}", preview)
console.print(table)

In [ ]:
# Sync the built ChromaDB back to Drive for persistence
console.print("[bold blue]Syncing ChromaDB to Google Drive...[/bold blue]")

if Path(DRIVE_OUTPUT_DIR).exists():
    shutil.rmtree(DRIVE_OUTPUT_DIR)

shutil.copytree(CHROMA_DIR, DRIVE_OUTPUT_DIR)

db_size = sum(f.stat().st_size for f in Path(DRIVE_OUTPUT_DIR).rglob("*"))
console.print(f"[green]✓ Saved to {DRIVE_OUTPUT_DIR} ({db_size/1024/1024:.1f} MB)[/green]")